# Time-Matched Post Window (17-day, comment-timestamp filter)

Tests whether a 17-day post-apology subset (Mimiminu-parity span) changes the post-window story.
Same per-comment scores as `04_sentiment_analysis.ipynb` — re-aggregation only, no model re-run.

**Key context:** the 10-video post list cannot be date-filtered (Bodeoum's first post video is 2024-07-06,
43 days after the apology), so the filter here is on **comment `timestamp`**, not video publish date.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd


In [ ]:
# Resolve repo root so the notebook runs from any cwd inside the repo
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "data").exists() and (REPO_ROOT / "03-analysis").exists():
        break
    REPO_ROOT = REPO_ROOT.parent

# 17-day parity span (Mimiminu post video span: 2025-11-25 -> 2025-12-12)
SPAN_DAYS = 17
CASES: list[str] = ["bodeoum_tv", "mimiminu"]

ANALYSIS_ROOT = REPO_ROOT / "03-analysis"
OUTPUT_DIR = ANALYSIS_ROOT / "outputs"

print("Repo root:", REPO_ROOT)
print("Span days:", SPAN_DAYS)


## Load scores + apology dates

Per-comment scores come from existing `sentiment_comments.csv` files.
Apology date = `create_time` of the `cancel` video record in each `<case>_youtube.jsonl`.

In [ ]:
def apology_date(case: str) -> str:
    """Read the cancel-video publish date (YYYY-MM-DD) from the raw JSONL."""
    path = REPO_ROOT / "data" / case / f"{case}_youtube.jsonl"
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            record = json.loads(line)
            if record.get("record_type") == "video" and record.get("window_label") == "cancel":
                return record["create_time"][:10]
    raise ValueError(f"no cancel video for {case}")


def load_scored(case: str) -> pd.DataFrame:
    """Load existing per-comment sentiment scores (model output is reused as-is)."""
    df = pd.read_csv(OUTPUT_DIR / case / "sentiment_comments.csv")
    df["comment_date"] = pd.to_datetime(df["timestamp"], unit="s", utc=True).dt.date.astype(str)
    return df


# Per-case apology anchor and 17-day cutoff, derived not hardcoded
windows = {
    case: (apo := apology_date(case), str((pd.to_datetime(apo) + pd.Timedelta(days=SPAN_DAYS)).date()))
    for case in CASES
}
scored = {case: load_scored(case) for case in CASES}
for case, (apo, cut) in windows.items():
    print(f"{case}: apology={apo} cutoff={cut} scored_comments={len(scored[case])}")


## Comment timestamp coverage

Checks whether the corpus holds event-contemporaneous comments at all —
collection ran ~2026-04, so comment timestamps may reflect scrape era, not the scandal timeline.

In [ ]:
# Timestamp span per window: min/max comment date and count
coverage_rows: list[dict[str, object]] = []
for case in CASES:
    df = scored[case]
    for window, grp in df.groupby("window"):
        coverage_rows.append(
            {
                "case": case,
                "window": window,
                "n_comments": len(grp),
                "min_comment_date": grp["comment_date"].min(),
                "max_comment_date": grp["comment_date"].max(),
            }
        )
coverage_df = pd.DataFrame(coverage_rows).sort_values(["case", "window"]).reset_index(drop=True)
coverage_df


## 17-day filter + re-aggregation

Same aggregation as `04_sentiment_analysis.ipynb`: `mean(sentiment_score)` plus label shares.
Guarded for empty subsets — an empty 17-day window is itself the finding.

In [ ]:
def summarize(sub: pd.DataFrame) -> dict[str, float]:
    """Window-level aggregation identical to 04_sentiment_analysis.ipynb."""
    n = len(sub)
    out: dict[str, float] = {"n_comments": n}
    if n == 0:
        return out | {"mean_sentiment": float("nan"), "pct_positive": float("nan"), "pct_negative": float("nan"), "pct_neutral": float("nan")}
    return out | {
        "mean_sentiment": float(sub["sentiment_score"].mean()),
        "pct_positive": float((sub["sentiment_label"] == "POSITIVE").mean() * 100),
        "pct_negative": float((sub["sentiment_label"] == "NEGATIVE").mean() * 100),
        "pct_neutral": float((sub["sentiment_label"] == "NEUTRAL").mean() * 100),
    }


# Filter comments to apology..cutoff and re-aggregate; anchors from existing window means
result_rows: list[dict[str, object]] = []
for case in CASES:
    apo, cut = windows[case]
    df = scored[case]
    sub = df[(df["comment_date"] >= apo) & (df["comment_date"] <= cut)]
    anchors = pd.read_csv(OUTPUT_DIR / case / "sentiment_by_window.csv").set_index("window")
    row = {"case": case, "filter_start": apo, "filter_end": cut}
    row.update({f"subset_{k}": v for k, v in summarize(sub).items()})
    row.update(
        {
            "full_post_mean": float(anchors.loc["post", "mean_sentiment"]),
            "pre_mean": float(anchors.loc["pre", "mean_sentiment"]),
            "cancel_mean": float(anchors.loc["cancel", "mean_sentiment"]),
        }
    )
    result_rows.append(row)
    print(f"{case}: subset n={len(sub)} | slots={sorted(sub['slot'].unique().tolist())[:10]}")
results_df = pd.DataFrame(result_rows)
results_df


## Save outputs

In [ ]:
# Persist coverage + subset summary for write-up use
coverage_path = OUTPUT_DIR / "comment_timestamp_coverage.csv"
results_path = OUTPUT_DIR / "time_matched_post_17d.csv"
coverage_df.to_csv(coverage_path, index=False)
results_df.to_csv(results_path, index=False)
print("Saved:", coverage_path)
print("Saved:", results_path)


## Findings

- The 17-day comment-timestamp subset is **empty for both cases** (n=0): no scored comment falls
  within apology..cutoff. Bodeoum cancel/post comments start 2025-04-20 (~11 months post-apology);
  Mimiminu cancel/post comments start 2025-12-20 (~4 weeks post-apology).
- Comment timestamps reflect the ~2026-04 collection scrape, not the scandal timeline — the corpus
  cannot support event-contemporaneous fine-grained timing analysis. Only pre-window videos carry
  contemporaneous (2024 / late-2025) comments.
- Reported window means (Bodeoum post 0.9001 vs pre 0.8132; Mimiminu post 0.8020 vs pre 0.7733)
  therefore compare scrape-era comment pools, not immediate-aftermath discourse. Candidate Limitations
  note; no existing result is invalidated.